## Metadata-Based Recommender

In [1]:
import os
import ast
import pandas as pd
import numpy as np
import joblib
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

os.chdir(r"C:\Users\Saeed\Desktop\neural_network_project")

PROCESSED_DIR = "data/processed"
MODELS_DIR    = "models/metadata"
os.makedirs(MODELS_DIR, exist_ok=True)

print("Ready!")

Ready!


In [2]:
df = pd.read_csv(f"{PROCESSED_DIR}/metadata.csv")

# تحويل الـ lists من string لـ list حقيقية
def parse_list(val):
    try:
        return ast.literal_eval(val)
    except:
        return []

df["genres_list"]   = df["genres_list"].apply(parse_list)
df["cast_list"]     = df["cast_list"].apply(parse_list)
df["keywords_list"] = df["keywords_list"].apply(parse_list)
df["director_clean"] = df["director_clean"].fillna("")

print(f"Loaded: {len(df)} movies")
df.head(3)

Loaded: 9988 movies


,movie_id,title,genres_list,cast_list,director_clean,keywords_list,metadata_soup,poster_path
0,983044,The Arctic Convoy,"[war, drama]","[tobias_santelmann, anders_baasmo_christiansen...",henrik_martin_dahlsbakken,"[world_war_ii, based_on_true_story, struggle_f...",war drama tobias_santelmann anders_baasmo_chri...,https://image.tmdb.org/t/p/w500/7qtd6xschdz7GB...
1,5,Four Rooms,[comedy],"[tim_roth, jennifer_beals, david_proval, ione_...",robert_rodriguez,"[hotel, new_year's_eve, witch, bet, sperm, hot...",comedy tim_roth jennifer_beals david_proval ro...,https://image.tmdb.org/t/p/w500/75aHn1NOYXh4M7...
2,11,Star Wars,"[adventure, action, science_fiction]","[mark_hamill, harrison_ford, carrie_fisher, pe...",george_lucas,"[empire, galaxy, rebellion, android, hermit, s...",adventure action science_fiction mark_hamill h...,https://image.tmdb.org/t/p/w500/6FfCtAuVAW8XJj...


In [3]:
def make_soup(row):
    genres   = " ".join(row["genres_list"])
    cast     = " ".join(row["cast_list"][:3])        # أول 3 ممثلين
    director = " ".join([row["director_clean"]] * 3) # المخرج وزن أعلى
    keywords = " ".join(row["keywords_list"][:5])

    # وزن الـ genres أعلى
    return f"{genres} {genres} {director} {cast} {keywords}"

df["soup"] = df.apply(make_soup, axis=1)

print("Sample soup:")
print(df.iloc[2]["soup"])

Sample soup:
adventure action science_fiction adventure action science_fiction george_lucas george_lucas george_lucas mark_hamill harrison_ford carrie_fisher empire galaxy rebellion android hermit


In [4]:
print("Building Count matrix...")

# CountVectorizer أفضل من TF-IDF للـ metadata
count = CountVectorizer(
    stop_words = "english",
    min_df     = 1,
    ngram_range = (1, 1)
)

count_matrix = count.fit_transform(df["soup"])

print(f"Matrix shape : {count_matrix.shape}")
print(f"Memory       : {count_matrix.data.nbytes / 1024 / 1024:.1f} MB")

Building Count matrix...
Matrix shape : (9988, 22486)
Memory       : 0.9 MB


In [5]:
print("Computing cosine similarity...")

cosine_sim_meta = cosine_similarity(count_matrix, count_matrix)

print(f"Shape  : {cosine_sim_meta.shape}")
print(f"Memory : {cosine_sim_meta.nbytes / 1024 / 1024:.1f} MB")

Computing cosine similarity...
Shape  : (9988, 9988)
Memory : 761.1 MB


In [6]:
indices_meta = pd.Series(df.index, index=df["title"].str.lower()).drop_duplicates()

def get_metadata_recommendations(title: str, n: int = 10) -> pd.DataFrame:
    """ترشيح بناءً على المخرج والممثلين والنوع"""
    
    title_lower = title.lower().strip()
    
    if title_lower not in indices_meta:
        matches = [t for t in indices_meta.index if title_lower in t]
        if not matches:
            print(f"'{title}' not found!")
            return pd.DataFrame()
        title_lower = matches[0]
        print(f"Using: '{title_lower}'")
    
    idx = indices_meta[title_lower]
    
    sim_scores = sorted(
        enumerate(cosine_sim_meta[idx]),
        key=lambda x: x[1],
        reverse=True
    )[1:n+1]
    
    movie_indices = [i[0] for i in sim_scores]
    scores        = [round(i[1], 4) for i in sim_scores]
    
    result = df.iloc[movie_indices][["movie_id","title","genres_list","director_clean"]].copy()
    result["similarity_score"] = scores
    result["genres"] = result["genres_list"].apply(lambda x: "|".join(x))
    
    return result[["movie_id","title","genres","director_clean","similarity_score"]].reset_index(drop=True)

print("get_metadata_recommendations() ready!")

get_metadata_recommendations() ready!


In [7]:
test_movies = ["The Dark Knight", "Toy Story", "The Godfather", 
               "Inception", "Pulp Fiction"]

print("=== Metadata Recommendations ===\n")
for movie in test_movies:
    recs = get_metadata_recommendations(movie, n=5)
    if not recs.empty:
        print(f"[{movie}]")
        for _, row in recs.iterrows():
            print(f"  → {row['title']:40s} dir: {row['director_clean']:25s} score: {row['similarity_score']:.4f}")
        print()

=== Metadata Recommendations ===

[The Dark Knight]
  → The Dark Knight Rises                    dir: christopher_nolan         score: 0.7435
  → Batman Begins                            dir: christopher_nolan         score: 0.6897
  → Insomnia                                 dir: christopher_nolan         score: 0.5862
  → Tenet                                    dir: christopher_nolan         score: 0.5670
  → Memento                                  dir: christopher_nolan         score: 0.4734

[Toy Story]
  → Cars                                     dir: john_lasseter             score: 0.8182
  → Cars 2                                   dir: john_lasseter             score: 0.7879
  → Toy Story 2                              dir: john_lasseter             score: 0.7758
  → A Bug's Life                             dir: john_lasseter             score: 0.7155
  → Mater and the Ghostlight                 dir: john_lasseter             score: 0.6674

[The Godfather]
  → The Godfather 

In [8]:
# شوف الفرق بين الموديلين
movie = "Inception"
print(f"=== Comparing models for: {movie} ===\n")

# Content-Based (من الـ notebook التاني)
content_df  = pd.read_csv("models/content_based/content_df.csv")
content_sim = np.load("models/content_based/cosine_sim.npy")
content_idx = pd.Series(content_df.index, 
                         index=content_df["title"].str.lower()).drop_duplicates()

idx = content_idx[movie.lower()]
content_scores = sorted(enumerate(content_sim[idx]), key=lambda x: x[1], reverse=True)[1:6]
content_recs   = [content_df.iloc[i[0]]["title"] for i in content_scores]

# Metadata
meta_recs = get_metadata_recommendations(movie, n=5)["title"].tolist()

print(f"{'Content-Based':^35} {'Metadata-Based':^35}")
print("-" * 70)
for c, m in zip(content_recs, meta_recs):
    print(f"{c:35s} {m:35s}")

=== Comparing models for: Inception ===

           Content-Based                      Metadata-Based           
----------------------------------------------------------------------
Interstellar                        Tenet                              
Tenet                               Interstellar                       
The Prestige                        Ra.One                             
Following                           Journey 2: The Mysterious Island   
Don Jon                             The Dark Knight                    


In [9]:
print("Saving model...")

joblib.dump(count,          f"{MODELS_DIR}/count_vectorizer.pkl")
np.save(f"{MODELS_DIR}/cosine_sim_meta.npy", cosine_sim_meta)
indices_meta.to_pickle(f"{MODELS_DIR}/indices_meta.pkl")
df.to_csv(f"{MODELS_DIR}/metadata_df.csv", index=False)

print("Saved:")
for f in os.listdir(MODELS_DIR):
    size = os.path.getsize(f"{MODELS_DIR}/{f}") / 1024 / 1024
    print(f"  {f:35s} {size:.1f} MB")

Saving model...
Saved:
  cosine_sim_meta.npy                 761.1 MB
  count_vectorizer.pkl                0.4 MB
  indices_meta.pkl                    0.3 MB
  metadata_df.csv                     6.5 MB
